In [ ]:
import numpy as np
import pandas as pd
from multiprocessing import Pool, Value, get_context
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
def load_and_preprocess_data(filepath):
    """Load pre-normalized DNA data from CSV file."""
    try:
        # Read the CSV file
        dna_df = pd.read_table(filepath, header=None)

        # Split the comma-separated values and convert to float
        if isinstance(dna_df[0].iloc[0], str):
            dna_df = pd.DataFrame([
                list(map(float, row.split(',')))
                for row in dna_df[0]
            ])

        return dna_df

    except Exception as e:
        print(f"Error loading data: {str(e)}")
        raise

In [ ]:
def create_covariates(n_samples, degree=3):
    """
    Create polynomial covariates for confounding correction.
    Based on sample numbers (patients).
    """
    # Create sample index
    sample_idx = np.arange(n_samples)

    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    covar = poly.fit_transform(sample_idx.reshape(-1, 1))

    # Add potential technical covariates here if needed
    # For example: batch effects, processing date, etc.
    # covar = np.column_stack([covar, batch_indicators])

    return covar

In [ ]:
def correct_confounding(DNA, covariates):
    """
    Perform confounding correction on the entire dataset at once.
    Handles cross-patient effects.
    """
    # Initialize regression model
    lm = LinearRegression(fit_intercept=True)

    # Correct each gene
    corrected_data = np.zeros_like(DNA)
    for j in range(DNA.shape[1]):
        # Fit regression model
        lm.fit(covariates, DNA[:, j])

        # Calculate residuals
        corrected_data[:, j] = DNA[:, j] - lm.predict(covariates)

        if j % 1000 == 0:
            print(f"Processed {j}/{DNA.shape[1]} genes")

    return corrected_data

In [ ]:
from google.colab import files

def main():
    # Upload input file
    print("Please upload your methylation CSV file (DNAm_processed_imputed.csv)...")
    uploaded = files.upload()
    input_file = list(uploaded.keys())[0]

    # Load data
    print("Loading DNA data...")
    DNA = load_and_preprocess_data(input_file)

    # Create covariates matrix based on number of patients
    print("Creating covariates...")
    covariates = create_covariates(DNA.shape[0])  # Using shape[0] for number of patients

    print("Starting confounding correction...")
    try:
        # Perform correction
        corrected_data = correct_confounding(DNA.to_numpy(), covariates)

        # Convert to DataFrame and save
        output_file = "DNA_corrected.csv"
        res_df = pd.DataFrame(
            corrected_data,
            index=DNA.index,
            columns=DNA.columns
        )
        res_df.to_csv(output_file, sep=',', index_label='TargetID')

        # Download the file
        files.download(output_file)

        print("\nResults preview:")
        print(res_df.iloc[:5, :5])

    except Exception as e:
        print(f"Error during processing: {str(e)}")
        raise

if __name__ == '__main__':
    main()

Please upload your methylation CSV file (DNAm_processed_imputed.csv)...


Saving DNAm_processed_imputed.csv to DNAm_processed_imputed.csv
Loading DNA data...
Creating covariates...
Starting confounding correction...
Processed 0/2000 genes
Processed 1000/2000 genes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Results preview:
          0         1         2         3         4
0 -0.202205 -0.416659 -0.325117  0.515517 -0.301613
1 -0.195217 -0.034789  0.391529 -0.367490  0.375760
2 -0.201264 -0.381300 -0.253358  0.445732 -0.297733
3 -0.212831  0.534786  0.530133 -0.376250 -0.302242
4  0.559756  0.233778  0.229682  0.149432  0.433567
